# Reproducing Zhang, Gao, Cukur & Gallant (2021) from CRCNS vim-4

Paper: Zhang T, Gao JS, Cukur T, Gallant JL (2021). Voxel-Based State Space Modeling Recovers
Task-Related Cognitive States in Naturalistic fMRI Experiments. Frontiers in Neuroscience
14:565976. doi:10.3389/fnins.2020.565976.

Claims: `claims/zhang-2021.yaml`, written from the version of record (PMC8145286) and its
supplement (Data_Sheet_1.DOCX), with the paper's code, gallantlab/state-space at commit 6ffb3c3.

Dataset: CRCNS vim-4, "Whole-brain BOLD activity recorded by fMRI during a visual attention task
and a video game task", doi:10.6080/K0668BDF, streamed from the public bucket
`s3://crcnsarchive/vim-4/live/`. Version: `unversioned`. CRCNS data sets have no versions; the
files read are those under `vim-4/live/` as of 2026-11-14 (data description version 0.8 of
2026-04-06), which is the only published copy of the data files (`vim-4/old/` holds only older
versions of the description). The paper cites the data set by its DOI, with no version. The MD5
sums of the files read, from the data set's `checksums.md5`, are:

| file | MD5 |
|---|---|
| VisualAttention/Data/deconv/S1.hdf | 5989fd2bc00fadf058eac9e7e742ad7d |
| VisualAttention/Data/deconv/S2.hdf | 94a06a22a634efb4370b3061b532744f |
| VisualAttention/Data/deconv/S3.hdf | 16758cb38128a43182c495660ac77eb6 |
| VisualAttention/Data/deconv/S5.hdf | 782fcf8c74b82812fb9fb8e7e106b285 |
| VisualAttention/Data/deconv/S6.hdf | 57fc3c680adac346ca41461e1a891592 |
| VisualAttention/Stimuli/states.hdf | 52e2cee2155281256192c2e9ded39383 |
| VisualAttention/Stimuli/semantics.hdf | 54143a9de19a34d647856693588dfc9d |
| VideoGame/Data/S1.npy | c0ea35bd959465d7e03cf5b369002640 |
| VideoGame/Data/S7.npy | 0ed00d89f585636b9091d56fc1a9ed81 |
| VideoGame/Stimuli/S1.hdf | d66b6a346ebfa13948b71da7a8d810a8 |
| VideoGame/Stimuli/S7.hdf | b5b5bb06946d4d6df1eb9c813d3f5825 |

The notebook prints the bucket's current sums for these files below, so a change can be seen.
Only the layouts of the other files (non-deconvolved BOLD, mappers) are read.

| claim | tested here |
|---|---|
| fig1-csi-s2 | tested |
| att-csi-group | tested (5 of 6 subjects; S4 is not released) |
| supp-csi-per-subject | tested (S4 left out, as the claim says) |
| att-passive-lower | tested (5 of 6 subjects) |
| att-classification | tested (5 of 6 subjects) |
| att-weights-cortex | blocked: the paper's functional ROIs (localizer and retinotopy data) are not public |
| game-csi | tested |
| game-mutual-negative-projection | tested |
| fig3b-state-relations | tested |
| game-classification | tested |
| game-weights-two-groups | blocked: the paper's ROIs are not public, and S1's game data have no matching mapper |

Method: the helper `statespace.py` says where each parameter comes from. The state space is the
code's `extract_state_space_positions`, computed exactly in the space of the 24 principal
components (checked against the code itself below).

In [ ]:
import time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import statespace as ss

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
def mem_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6

## Data check

In [ ]:
# Files in the data set and their current MD5 sums (checksums.md5 in the bucket)
listing = requests.get("https://crcnsarchive.s3-us-west-2.amazonaws.com/?list-type=2&prefix=vim-4/").text
import re
files = [(k, int(s)) for k, s in re.findall(r"<Key>(.*?)</Key>.*?<Size>(\d+)</Size>", listing) if not k.endswith("/")]
print(f"{len(files)} files under vim-4/")
for k, s in files:
    print(f"  {s/1e6:10.2f} MB  {k}")
md5 = dict(l.split()[::-1] for l in requests.get(ss.BASE + "checksums.md5").text.strip().splitlines())
read_files = ["VisualAttention/Data/deconv/S%d.hdf" % i for i in (1, 2, 3, 5, 6)] + [
    "VisualAttention/Stimuli/states.hdf", "VisualAttention/Stimuli/semantics.hdf",
    "VideoGame/Data/S1.npy", "VideoGame/Data/S7.npy", "VideoGame/Stimuli/S1.hdf", "VideoGame/Stimuli/S7.hdf"]
for f in read_files:
    print(md5[f], f)

In [ ]:
# Layout of every data file: datasets, shapes and dtypes (no data read)
for f in ["VisualAttention/Data/S%d.hdf" % i for i in (1, 2, 3, 5, 6)] + \
         ["VisualAttention/Data/deconv/S%d.hdf" % i for i in (1, 2, 3, 5, 6)] + \
         ["VisualAttention/Stimuli/states.hdf", "VisualAttention/Stimuli/semantics.hdf",
          "VideoGame/Stimuli/S1.hdf", "VideoGame/Stimuli/S7.hdf"] + \
         ["Mappers/S%d.hdf" % i for i in range(1, 8)]:
    for name, shape, dtype in ss.hdf_layout(f):
        print(f"{f:42s} {name:28s} {str(shape):18s} {dtype}")
for s in ss.GAME_SUBJECTS:
    print(f"VideoGame/Data/{s}.npy", ss.npy_layout(f"VideoGame/Data/{s}.npy"))

In [ ]:
# Attention task labels: states.hdf, and the semantic labels they should follow
st = ss.load_attention_states()
for c in ss.CONDITIONS:
    v = st[c]
    print(c, v.shape, "| human presence values", np.unique(v[:, 0]), "| vehicle", np.unique(v[:, 1]),
          "| attention column values", np.unique(v[:, 2]),
          "| presence counts (neither, humans, vehicles, both)", np.bincount(ss.presence_code(v), minlength=4))
print("presence columns identical across the three conditions:",
      np.array_equal(st["human"][:, :2], st["vehicle"][:, :2]) and np.array_equal(st["human"][:, :2], st["passive"][:, :2]))
X_att = np.vstack([st[c] for c in ss.CONDITIONS])
pcode = ss.presence_code(X_att)

sem = ss.read_hdf("VisualAttention/Stimuli/semantics.hdf")
sel = sem["passive_selection_indices"]
print("passive_selection_indices: %d sections, %d s selected in total (900 TRs x 2 s = 1800 s), range %d to %d s"
      % (len(sel), (sel[:, 1] - sel[:, 0]).sum(), sel.min(), sel.max()))
# The code's subsample_passive_data applied to the passive semantic labels should give the attentive labels
from scipy import signal
up = signal.resample(sem["passive"], 7200)
down = signal.resample(np.vstack([up[a:b] for a, b in sel]), 900)
print("correlation of the subsampled passive semantic labels with the attentive ones: %.3f"
      % np.corrcoef(down.ravel(), sem["attentive"].ravel())[0, 1])
cats = requests.get(ss.BASE + "docs/attention_categories.txt").text.split()
pres = sem["attentive"] > 0
for col, name in [(0, "human"), (1, "vehicle")]:
    y = st["human"][:, col]
    r = np.array([np.corrcoef(pres[:, j], y)[0, 1] if pres[:, j].std() > 0 else 0 for j in range(pres.shape[1])])
    top = np.argsort(r)[::-1][:3]
    print(f"{name} presence: best matching semantic categories", [(cats[j], round(r[j], 2)) for j in top])

In [ ]:
# Load each attention subject's deconvolved data (stacked passive, human, vehicle), keep the
# 2700 x 2700 Gram matrix, and run the paper's code itself on the full voxel data for comparison.
att = {}
for s in ss.ATT_SUBJECTS:
    t = time.time()
    d = ss.load_attention_gram(s, X_check=X_att)
    G = d["G"]
    # per-condition mean and SD of the voxels, from the Gram matrix diagonal blocks is not enough,
    # so recompute from the summaries: sum of squares per TR
    Z, _, ve = ss.pca_scores(G, np.arange(len(G)))
    P = np.column_stack([X_att, np.ones(len(X_att))])
    coef = np.linalg.pinv(P.T @ P) @ (P.T @ Z)
    pos_rw = ss.att_positions(Z, X_att, "qr")
    cp = d["code_positions"]
    corr = [abs(np.corrcoef(pos_rw[:, j], cp[:, j])[0, 1]) for j in range(3)]
    # same-frame TRs across conditions: correlation of their voxel patterns (duplicate check)
    nrm = np.sqrt(np.diag(G))
    same = {f"{a}-{b}": float(np.mean([G[i + 900 * ia, i + 900 * ib] / nrm[i + 900 * ia] / nrm[i + 900 * ib] for i in range(900)]))
            for ia, a in enumerate(ss.CONDITIONS) for ib, b in enumerate(ss.CONDITIONS) if ia < ib}
    att[s] = dict(G=G, Z=Z, code_pos=cp, ve=ve, n_vox=d["n_vox"])
    print(f"{s}: {d['n_vox']} voxels ({d['n_zero_vox']} constant, {d['n_nan']} NaN), voxel means {d['vox_mean_range'][0]:.3f} to "
          f"{d['vox_mean_range'][1]:.3f}, SDs {d['vox_sd_range'][0]:.3f} to {d['vox_sd_range'][1]:.3f}; "
          f"24 PCs explain {ve:.1%}; |weights| in PC space human {np.linalg.norm(coef[0]):.3g}, vehicle "
          f"{np.linalg.norm(coef[1]):.3g}, attention {np.linalg.norm(coef[2]):.3g}; |r| rewrite vs code per axis "
          f"{np.round(corr, 4)}; mean r of same-frame TRs {same}; {time.time()-t:.0f} s, peak memory {mem_gb():.1f} GB")
nv = [att[s]["n_vox"] for s in att]
print(f"voxels per subject: mean {np.mean(nv):.0f}, SD {np.std(nv, ddof=1):.0f} over {len(nv)} subjects (paper: 48,673 +/- 4,382 over six)")

In [ ]:
# Per-condition means of the voxels: the attentive conditions are z-scored per voxel within the condition,
# so the attention variable (-1 human, 0 passive, +1 vehicle) carries no mean difference between them.
for s in ss.ATT_SUBJECTS:
    G = att[s]["G"]
    # the mean over voxels of each TR's squared norm, per condition, and the norm of each condition's mean pattern
    msq = [np.mean(np.diag(G)[sl]) for sl in ss.cond_slices()]
    mean_norm = [np.sqrt(max(G[sl, sl].sum(), 0.0)) / 900 for sl in ss.cond_slices()]  # rounding can make a zero sum slightly negative
    print(f"{s}: mean squared norm per TR {np.round(msq, 0)}; norm of each condition's mean pattern "
          f"(passive, human, vehicle) {np.round(mean_norm, 4)}")

In [ ]:
# Clock check for the attention task: CSI of the presence clusters when the presence labels are
# shifted in time within each condition (refitted at each lag). Deconvolved data should peak at lag 0.
lags = [-6, -3, -2, -1, 0, 1, 2, 3, 6, 20]
rng = np.random.default_rng(100)
rows = []
for s in ss.ATT_SUBJECTS:
    Z = att[s]["Z"]
    for lag in lags:
        Xs = X_att.copy()
        for sl in ss.cond_slices():
            Xs[sl, :2] = np.roll(X_att[sl, :2], lag, axis=0)
        pos = ss.att_positions(Z, Xs, "2d")
        rows.append(dict(subject=s, lag=lag, csi=ss.att_csis(pos, ss.presence_code(Xs), rng).mean()))
shift = pd.DataFrame(rows).pivot(index="subject", columns="lag", values="csi")
print("mean CSI over conditions, human-vehicle plane, by label lag in TRs (positive: labels moved later)")
print(shift.round(3))
print("lag of the peak per subject:", shift.idxmax(axis=1).to_dict())

In [ ]:
# Video game data
game = {}
for s in ss.GAME_SUBJECTS:
    t = time.time()
    g = ss.load_game(s)
    G = g["G"]
    Z, _, ve = ss.pca_scores(G, np.arange(len(G)))
    g["Z"], g["ve"] = Z, ve
    game[s] = g
    X = g["X"]
    nst = X.sum(1).astype(int)
    print(f"{s}: {X.shape[0]} TRs ({X.shape[0]*2.0045/60:.1f} min at TR 2.0045 s), {g['n_vox']} voxels "
          f"({g['n_zero_vox']} constant, {g['n_nan']} NaN), voxel means {g['vox_mean_range'][0]:.3f} to {g['vox_mean_range'][1]:.3f}, "
          f"SDs {g['vox_sd_range'][0]:.3f} to {g['vox_sd_range'][1]:.3f}; 24 PCs explain {ve:.1%}; {time.time()-t:.0f} s")
    print("   state labels (dataset order):", g["labels"])
    print("   TRs per state:", dict(zip(g["labels"], X.sum(0).astype(int))))
    print("   TRs with 0, 1, 2, 3 states:", np.bincount(nst, minlength=4))
    print("   TRs with exactly one state, per state:", dict(zip(g["labels"], X[nst == 1].sum(0).astype(int))))
    print("   safe explore and engage close together in", int(((X[:, g['labels'].index('safe explore')] > 0) & (X[:, g['labels'].index('engage close')] > 0)).sum()), "TRs")
mapper_vox = {f"S{i}": ss.read_hdf(f"Mappers/S{i}.hdf", ["vox_to_fsavg_left_shape"])["vox_to_fsavg_left_shape"][1] for i in range(1, 8)}
print("voxels per mapper:", mapper_vox)
print("game voxels:", {s: game[s]["n_vox"] for s in ss.GAME_SUBJECTS}, "| attention voxels (deconv files):",
      {s: att[s]["n_vox"] for s in att})
print(f"elapsed {time.time()-T0:.0f} s, peak memory {mem_gb():.1f} GB")

In [ ]:
# Clock check for the video game: CSI (one cluster per state) with the state matrix shifted in time
rng = np.random.default_rng(101)
rows = []
for s in ss.GAME_SUBJECTS:
    X, names = ss.game_matrix(game[s]["X"], game[s]["labels"], "dataset")
    for lag in [-10, -4, -2, -1, 0, 1, 2, 3, 4, 6, 10]:
        Xs = np.roll(X, lag, axis=0)
        mem, _, _ = ss.game_members(Xs, names, "a")
        rows.append(dict(subject=s, lag=lag, csi=ss.csi(ss.positions(game[s]["Z"], Xs), mem, rng)))
gshift = pd.DataFrame(rows).pivot(index="subject", columns="lag", values="csi")
print(gshift.round(3))
print("lag of the peak per subject:", gshift.idxmax(axis=1).to_dict())

Checks of the description against the data, from the outputs above:

- The deconvolved attention files hold 900 TRs per condition for S1, S2, S3, S5 and S6 (30 min each
  at a 2 s TR, three 10-min runs as the paper says); S4 is absent, as the description says. The
  passive deconvolved data are already subsampled to the 900 TRs of the attentive movies. The
  selection indices cover 1800 s, and applying the code's subsampling to the passive semantic
  labels reproduces the attentive labels (correlation printed above), which is consistent with
  the passive data and labels sharing the attentive clock.
- `states.hdf` has the same presence labels in all three conditions; the third column codes the
  condition as -1 (attend humans), 0 (passive) and +1 (attend vehicles). The description does not
  give this coding. The vehicle column follows the semantic category `vehicle.n.01` exactly, and
  the human column follows `person.n.01` closely.
- The attentive conditions are z-scored per voxel within each condition (voxel means of zero, so
  the norm of each attentive condition's mean pattern is zero), while the passive TRs, subsampled
  after z-scoring, are not. With the attention variable coded -1, 0, +1 and an intercept, its OLS
  weight is the difference of the two attentive means, which is zero: the printed attention weight
  is at the level of rounding (1e-13), so the third axis of the state space is set by rounding
  noise. This is analysed in each attention claim below.
- The PC-space rewrite gives the code's human and vehicle axes exactly (|r| = 1.0000); its third
  axis differs, being noise in both. 24 PCs explain about 85 to 90% of the variance here, against
  about 70% in the paper.
- Labels and BOLD share a clock as far as shifted labels show: the attention CSI peaks at lag 0,
  as deconvolved data should, and the game CSI peaks within one TR of lag 0. This is consistent
  with, not proof of, aligned labels.
- Video game: S7 has 1320 TRs (44 min, the paper's 45 min). S1 has 1950 TRs, 65 min, where the
  paper says one subject played 90 min in six 15-min runs. The S7 data are float32 and not exactly
  z-scored per voxel over the session; S1's are. No flash TRs in S7, as the paper says. The S1
  game data have 73,594 voxels, which matches no mapper (S1's mapper has 51,608, as the S1
  attention data), so S1's game weights cannot be put on fsaverage with the files given. S7's
  data match S7's mapper. The game state columns are not in the order of Table 1.
- No recording is deposited twice: same-frame TRs of different conditions are only weakly
  correlated (printed above), and the subjects' voxel counts all differ.

## Attention task: CSI per subject and condition

Shared by fig1-csi-s2, att-csi-group and supp-csi-per-subject. The CSI of the four presence
clusters within each condition, in the state space fitted once on the three conditions stacked.
Because the third axis is undefined in these data, three readings are computed:

- R3, "code": the paper's code run as written on the full voxel data (its third axis is whatever
  rounding gives on this machine);
- R2, "random third axis": the median over 100 random third axes, orthogonal to the human and
  vehicle axes in the space of the 24 PCs (the distribution of what an arbitrary third axis gives);
- R1, "2D": the human-vehicle plane alone.

CSI values are means over 10 Monte Carlo seeds. The null refits the state space with permuted
presence labels, 1000 times per reading of the claims file: (a) the (human, vehicle) rows shuffled
over all TRs, and (b) circularly shifted by a random lag within each condition. Each null draw uses
a random third axis (for R3 and R2) or the plane (for R1) and one Monte Carlo seed.

In [ ]:
N_PERM = 1000
rng = np.random.default_rng(1)
csi_obs, csi_null, csi_seedsd = {}, {}, {}
for s in ss.ATT_SUBJECTS:
    t = time.time()
    Z = att[s]["Z"]
    r3 = np.array([ss.att_csis(att[s]["code_pos"], pcode, np.random.default_rng(k)) for k in range(10)])
    r1 = np.array([ss.att_csis(ss.att_positions(Z, X_att, "2d"), pcode, np.random.default_rng(k)) for k in range(10)])
    r2 = np.array([ss.att_csis(ss.att_positions(Z, X_att, "random", rng), pcode, rng) for _ in range(100)])
    csi_obs[s] = {"R3": r3.mean(0), "R2": np.median(r2, 0), "R1": r1.mean(0),
                  "R2_lo": np.percentile(r2, 5, axis=0), "R2_hi": np.percentile(r2, 95, axis=0)}
    csi_seedsd[s] = {"R3": r3.std(0), "R1": r1.std(0)}
    for reading in "ab":
        n3, n2 = [], []
        for _ in range(N_PERM):
            Xp = ss.shuffle_presence(X_att, reading, rng)
            pc_p = ss.presence_code(Xp)
            n3.append(ss.att_csis(ss.att_positions(Z, Xp, "random", rng), pc_p, rng))
            n2.append(ss.att_csis(ss.att_positions(Z, Xp, "2d"), pc_p, rng))
        csi_null[s, reading, "3d"] = np.array(n3)
        csi_null[s, reading, "2d"] = np.array(n2)
    print(f"{s} done in {time.time()-t:.0f} s")

def pval(obs, null):
    return (1 + (null >= obs).sum(0)) / (len(null) + 1)

rows = []
for s in ss.ATT_SUBJECTS:
    for ci, c in enumerate(ss.CONDITIONS):
        row = dict(subject=s, condition=c, R3_code=csi_obs[s]["R3"][ci], R3_seed_sd=csi_seedsd[s]["R3"][ci],
                   R2_median=csi_obs[s]["R2"][ci], R2_5pct=csi_obs[s]["R2_lo"][ci], R2_95pct=csi_obs[s]["R2_hi"][ci],
                   R1_2d=csi_obs[s]["R1"][ci])
        for reading in "ab":
            row[f"null3d_max_{reading}"] = csi_null[s, reading, "3d"][:, ci].max()
            row[f"null2d_max_{reading}"] = csi_null[s, reading, "2d"][:, ci].max()
        rows.append(row)
csi_table = pd.DataFrame(rows)
print(csi_table.round(3).to_string())
print(f"null p-values are at most 1/{N_PERM+1} = {1/(N_PERM+1):.4f} when the observed CSI exceeds every null value")
print(f"elapsed {time.time()-T0:.0f} s, peak memory {mem_gb():.1f} GB")

### fig1-csi-s2

> In subject S2 the four presence states (neither, humans only, vehicles only, both) are
> significantly separated in the attention-task state space, with a cluster separation index of
> 0.75 for attend-humans, 0.69 for attend-vehicles and 0.67 for passive viewing.

In [ ]:
exp = {"human": (0.75, 0.075), "vehicle": (0.69, 0.069), "passive": (0.67, 0.067)}
t = csi_table[csi_table.subject == "S2"].set_index("condition")
for reading in ["R3_code", "R2_median", "R1_2d"]:
    null = "null2d" if reading == "R1_2d" else "null3d"
    for c in ["human", "vehicle", "passive"]:
        v = t.loc[c, reading]; e, tol = exp[c]
        print(f"{reading:9s} {c:8s} CSI {v:.3f} (expected {e} +/- {tol}; within: {abs(v-e) <= tol}); "
              f"above every null value: (a) {v > t.loc[c, null + '_max_a']} (max {t.loc[c, null + '_max_a']:.3f}), "
              f"(b) {v > t.loc[c, null + '_max_b']} (max {t.loc[c, null + '_max_b']:.3f})")
    ok = all(abs(t.loc[c, reading] - exp[c][0]) <= exp[c][1] for c in exp)
    print(f"  -> {reading}: all three within tolerance: {ok}")
print("R2 5 to 95% range over random third axes:", {c: (round(t.loc[c, 'R2_5pct'], 3), round(t.loc[c, 'R2_95pct'], 3)) for c in exp})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
cols = ["#ff5100", "#0084ff", "#91d90d", "#a200ff"]
pos = att["S2"]["code_pos"]
out = 0
for ax, (ci, c) in zip(axes, enumerate(ss.CONDITIONS)):
    sl = ss.cond_slices()[ci]
    p, k = pos[sl], pcode[sl]
    sx = 1 if p[k == 3, 0].mean() > p[k == 0, 0].mean() else -1   # the code's sign rule
    sy = 1 if p[k == 3, 1].mean() > p[k == 0, 1].mean() else -1
    for j in range(4):
        ax.scatter(sx * p[k == j, 0], sy * p[k == j, 1], s=4, alpha=0.4, c=cols[j], label=ss.PRESENCE_NAMES[j])
        ax.scatter(sx * p[k == j, 0].mean(), sy * p[k == j, 1].mean(), marker="+", s=300, c="k")
    out += int((np.abs(p[:, :2]) > 4).any(1).sum())
    ax.set(xlim=(-4, 4), ylim=(-4, 4), title=f"S2 {c}", xlabel="human presence", ylabel="vehicle presence")
axes[0].legend(markerscale=3, fontsize=7)
plt.tight_layout(); plt.show()
print(f"positions range {pos[:, :2].min():.2f} to {pos[:, :2].max():.2f}; {out} TRs outside the axes")

**Result: reproduced, resting on an assumption about the third axis.** With the paper's code (R3)
the S2 CSIs are 0.740 (attend-humans), 0.718 (attend-vehicles) and 0.676 (passive), all within
tolerance of 0.75, 0.69 and 0.67, and each exceeds every one of 1000 null values under both
permutation readings (null maxima 0.040 to 0.062 under (a), 0.322 to 0.352 under (b); p <= 0.001,
the bound 1000 permutations reach, not the paper's 1e-5). The median over random third axes (R2)
also matches (0.723, 0.718, 0.657). In the human-vehicle plane alone (R1) the attend-humans CSI is
0.664, outside tolerance. The third axis of the state space has no weight in these data, so the
values depend on it: over random third axes the attend-humans CSI ranges from 0.683 to 0.784 (5 to
95%).

### att-csi-group

> Across subjects the presence states are distinctly and significantly separated in the
> attention-task state space, with CSI 0.66 +/- 0.09 for attend-humans, 0.74 +/- 0.03 for
> attend-vehicles and 0.53 +/- 0.24 for passive viewing (mean +/- std).

In [ ]:
exp = {"human": (0.66, 0.09), "vehicle": (0.74, 0.03), "passive": (0.53, 0.24)}
fig_ref = {"human": np.mean([0.51, 0.75, 0.70, 0.66, 0.60]), "vehicle": np.mean([0.79, 0.69, 0.71, 0.74, 0.71]),
           "passive": np.mean([0.64, 0.67, 0.05, 0.59, 0.61])}
print("second reference, means of the paper's per-subject values over the same five subjects:",
      {c: round(v, 3) for c, v in fig_ref.items()})
for reading in ["R3_code", "R2_median", "R1_2d"]:
    null = "2d" if reading == "R1_2d" else "3d"
    for c in ["human", "vehicle", "passive"]:
        ci = ss.CONDITIONS.index(c)
        v = csi_table[csi_table.condition == c][reading].values
        e, tol = exp[c]
        key = reading.split("_")[0]
        sig_a = {r: all(pval(csi_obs[s][key][ci], csi_null[s, r, null][:, ci]) <= 1 / (N_PERM + 1)
                        for s in ss.ATT_SUBJECTS) for r in "ab"}
        grp_null = {r: np.mean([csi_null[s, r, null][:, ci] for s in ss.ATT_SUBJECTS], axis=0) for r in "ab"}
        sig_b = {r: bool(v.mean() > grp_null[r].max()) for r in "ab"}
        print(f"{reading:9s} {c:8s} mean {v.mean():.3f} +/- {v.std(ddof=1):.3f} (SD over 5) (expected {e} +/- {tol}; "
              f"within: {abs(v.mean()-e) <= tol}); every subject above every null value: {sig_a}; "
              f"mean above every mean null value: {sig_b} (max mean null a {grp_null['a'].max():.3f}, b {grp_null['b'].max():.3f})")

**Result: reproduced, resting on an assumption about the third axis.** Over the five released
subjects, R3 gives mean CSIs of 0.665 +/- 0.103 (attend-humans), 0.729 +/- 0.022 (attend-vehicles)
and 0.525 +/- 0.260 (passive), all within the paper's spreads of 0.66 +/- 0.09, 0.74 +/- 0.03 and
0.53 +/- 0.24; R2 gives 0.658, 0.728 and 0.523. Under group reading (b), each mean exceeds the
maximum of the mean null CSIs (at most 0.110). Under reading (a), every subject is significant in
the attentive conditions but not in passive, where S3's CSI (0.061) does not exceed its null. In
the plane alone (R1) the attend-vehicles mean, 0.675, is outside tolerance. The second reference,
the means of the paper's own per-subject values over these subjects (0.644, 0.728, 0.512), is also
close to the R3 means.

### supp-csi-per-subject

> In each subject and each attention condition the CSI is significant, with the values printed in
> Supplementary Figures 1 and 2.

In [ ]:
supp = {("S1", "human"): 0.51, ("S1", "vehicle"): 0.79, ("S1", "passive"): 0.64,
        ("S3", "human"): 0.70, ("S3", "vehicle"): 0.71, ("S3", "passive"): 0.05,
        ("S5", "human"): 0.66, ("S5", "vehicle"): 0.74, ("S5", "passive"): 0.59,
        ("S6", "human"): 0.60, ("S6", "vehicle"): 0.71, ("S6", "passive"): 0.61}
for reading in ["R3_code", "R2_median", "R1_2d"]:
    null = "null2d" if reading == "R1_2d" else "null3d"
    rows, n_in, n_sig = [], 0, {"a": 0, "b": 0}
    for (s, c), e in supp.items():
        if s not in ss.ATT_SUBJECTS:
            continue
        r = csi_table[(csi_table.subject == s) & (csi_table.condition == c)].iloc[0]
        tol = 0.1 * e
        inside = abs(r[reading] - e) <= tol + 1e-12
        n_in += inside
        key, ci = reading.split("_")[0], ss.CONDITIONS.index(c)
        nk = "2d" if reading == "R1_2d" else "3d"
        rows.append(dict(subject=s, condition=c, expected=e, measured=round(r[reading], 3), within=inside,
                         null_max_a=round(r[f"{null}_max_a"], 3), p_a=pval(csi_obs[s][key][ci], csi_null[s, "a", nk][:, ci]),
                         null_max_b=round(r[f"{null}_max_b"], 3), p_b=pval(csi_obs[s][key][ci], csi_null[s, "b", nk][:, ci])))
        for rd in "ab":
            n_sig[rd] += rows[-1][f"p_{rd}"] < 0.01
    print(reading)
    print(pd.DataFrame(rows).to_string(index=False))
    print(f"  {n_in} of 12 within tolerance; significant at p < 0.01 ({N_PERM} permutations): "
          f"reading a {n_sig['a']} of 12, reading b {n_sig['b']} of 12\n")
i = ss.CONDITIONS.index("passive")
print("S3 passive: R3 mean over 10 seeds %.3f (seed SD %.3f); R2 5-95%% range %.3f to %.3f"
      % (csi_obs["S3"]["R3"][i], csi_seedsd["S3"]["R3"][i], csi_obs["S3"]["R2_lo"][i], csi_obs["S3"]["R2_hi"][i]))

**Result: reproduced, resting on an assumption about the third axis.** With R3, 10 of the 12
per-subject CSIs are within 10% of the values in Supplementary Figures 1 and 2 (11 of 12 with R2,
10 of 12 with R1). The two outside are S3 passive (0.061 against 0.05, a tolerance of 0.005) and
S5 attend-humans (0.767 against 0.66, within tolerance under R2 at 0.694). Under permutation
reading (a) all 12 R3 CSIs are significant at p < 0.01: eleven exceed every null value
(p = 0.001) and S3 passive has p = 0.002. Under reading (b) S3 passive has p = 0.042, so 11 of 12
are significant. The R1 counts are printed above. S3's passive CSI is close to the floor of the
measure, as in the paper (0.05).

### att-passive-lower

> Attention increases state separation; the CSI under passive viewing is significantly lower than
> under the attentive conditions in five of the six subjects.

Readings of the claims file: (a) passive against each attentive condition, both differences
significant; (b) passive against the mean of the two attentive CSIs. Null: the condition labels
permuted between TRs, presence labels kept, either (i) refitting the state space each time, or
(ii) keeping the fitted positions. How the condition labels are permuted is not fixed; two schemes
are run: "matched", where at each movie time point the three TRs showing the same frame get the
three conditions in random order (each condition keeps the same presence labels), and "global", a
shuffle over all TRs. One-sided, passive lower; 1000 permutations, so a subject counts at the
paper's threshold when its difference exceeds every null value (p <= 0.001, the bound reached).
The observed CSIs are those of R3 (the code's positions); for (i) the refitted null uses the
code's QR, which now gives the third axis a real weight. R1 (the plane) is shown as a variant.

In [ ]:
rng = np.random.default_rng(2)
def stats_ab(c):  # c: CSI (passive, human, vehicle)
    return np.array([c[1] - c[0], c[2] - c[0], c[1:].mean() - c[0]])
pl_rows = []
for s in ss.ATT_SUBJECTS:
    Z = att[s]["Z"]
    for variant, obs_pos in [("R3", att[s]["code_pos"]), ("R1", ss.att_positions(Z, X_att, "2d"))]:
        obs_c = np.mean([ss.att_csis(obs_pos, pcode, np.random.default_rng(k)) for k in range(10)], axis=0)
        obs = stats_ab(obs_c)
        for scheme in ["matched", "global"]:
            for refit in ["i", "ii"]:
                null = []
                for _ in range(N_PERM):
                    mem = ss.shuffle_conditions(900, scheme, rng)
                    if refit == "i":
                        Xp = ss.members_to_X(X_att, mem)
                        p = ss.att_positions(Z, Xp, "qr" if variant == "R3" else "2d")
                    else:
                        p = obs_pos
                    null.append(stats_ab(ss.att_csis(p, pcode, rng, members=mem)))
                null = np.array(null)
                p_one = (1 + (null >= obs).sum(0)) / (N_PERM + 1)
                pl_rows.append(dict(subject=s, variant=variant, scheme=scheme, null=refit,
                                    passive=obs_c[0], human=obs_c[1], vehicle=obs_c[2],
                                    d_human=obs[0], d_vehicle=obs[1], d_mean=obs[2],
                                    p_human=p_one[0], p_vehicle=p_one[1], p_mean=p_one[2]))
pl = pd.DataFrame(pl_rows)
pmin = 1 / (N_PERM + 1)
pl["a_at_bound"] = (pl.p_human <= pmin) & (pl.p_vehicle <= pmin)
pl["b_at_bound"] = pl.p_mean <= pmin
pl["a_p05"] = (pl.p_human < 0.05) & (pl.p_vehicle < 0.05)
pl["b_p05"] = pl.p_mean < 0.05
print(pl.round(4).to_string(index=False))
cnt = pl.groupby(["variant", "scheme", "null"])[["a_at_bound", "b_at_bound", "a_p05", "b_p05"]].sum()
print("\nsubjects of 5 with passive significantly lower:")
print(cnt.to_string())
print(f"elapsed {time.time()-T0:.0f} s")

**Result: not reproduced.** At the threshold that 1000 permutations allow (the difference
exceeding every null value), the passive CSI is significantly lower than the attentive CSIs in 2
or 3 of the 5 subjects under every combination of reading and null run with the code's positions
(R3), and in 1 with the plane (R1), where the criterion needs 4 of 5. At p < 0.05 the counts are
at most 3 with R3 and at most 4 with R1 (reading b, matched permutation, refitted). S3 and S5 show
the effect clearly. In S1 and S6 the attend-humans CSI is below the passive one (S1 0.518 against
0.622; S6 0.603 against 0.652), as in the paper's own figure values for these subjects, and the
mean of the two attentive CSIs is within 0.006 of passive. S2's differences (0.065 and 0.043) reach
p = 0.002 to 0.075.

### att-classification

> Held-out TRs of the attention task are classified into the 12 labels (four presence states by
> three attention conditions) from their position in the learned state space at 48.4 +/- 7.6%
> across subjects, well above the chance of 8.3%.

10-fold cross-validation over the 2700 stacked TRs; in each fold the PCA and the state space are
fitted on the training TRs (Methods, Quantification, paragraph 2), the held-out TRs are projected
and z-scored with the training statistics, and assigned to the nearest of the 12 training
clusters. Readings: folds (a) contiguous tenths or (b) random TRs; nearest cluster by (a) the
Euclidean distance to the mean or (b) the density of the cluster's normal. Null: the training
TRs' state rows permuted and the full fit repeated, 1000 times. In the training folds the two
attentive conditions are no longer exactly zero-mean, so the third axis gets a small fitted weight.

In [ ]:
lab12 = pcode + 4 * np.repeat(np.arange(3), 900)
classes12 = np.arange(12)
N_PERM_CLF = 1000
rng = np.random.default_rng(3)
clf_rows, clf_null = [], {}
for s in ss.ATT_SUBJECTS:
    t = time.time()
    for scheme in ["contig", "random"]:
        fs = ss.fold_scores(att[s]["G"], ss.folds(2700, scheme, rng))
        for rule in ["euclid", "density"]:
            accs = []
            for tr, te, Ztr, Zte in fs:
                ptr, pte = ss.project_fold(Ztr, Zte, X_att[tr])
                accs.append((ss.nearest_cluster(ptr, lab12[tr], pte, classes12, rule) == lab12[te]).mean())
            null = np.zeros(N_PERM_CLF)
            for k in range(N_PERM_CLF):
                a = []
                for tr, te, Ztr, Zte in fs:
                    perm = rng.permutation(len(tr))
                    ptr, pte = ss.project_fold(Ztr, Zte, X_att[tr][perm])
                    a.append((ss.nearest_cluster(ptr, lab12[tr][perm], pte, classes12, rule) == lab12[te]).mean())
                null[k] = np.mean(a)
            clf_null[s, scheme, rule] = null
            clf_rows.append(dict(subject=s, folds=scheme, rule=rule, accuracy=np.mean(accs), fold_sd=np.std(accs),
                                 null_mean=null.mean(), null_max=null.max(), p=(1 + (null >= np.mean(accs)).sum()) / (N_PERM_CLF + 1)))
    print(f"{s} done in {time.time()-t:.0f} s")
clf = pd.DataFrame(clf_rows)
print((clf.assign(accuracy=100 * clf.accuracy, fold_sd=100 * clf.fold_sd, null_mean=100 * clf.null_mean,
                  null_max=100 * clf.null_max)).round(2).to_string(index=False))
print("\nacross subjects (percent):")
for (scheme, rule), g in clf.groupby(["folds", "rule"]):
    gnull = np.mean([clf_null[s, scheme, rule] for s in ss.ATT_SUBJECTS], axis=0)
    m = 100 * g.accuracy.mean()
    print(f"  folds {scheme:6s} {rule:7s}: {m:.1f} +/- {100*g.accuracy.std(ddof=1):.1f} (SD over 5); within 48.4 +/- 7.6: "
          f"{abs(m-48.4) <= 7.6}; chance under this procedure (mean null) {100*gnull.mean():.1f}; max of mean null {100*gnull.max():.1f}; "
          f"above every null mean: {m > 100*gnull.max()}")
print(f"elapsed {time.time()-T0:.0f} s")

**Result: reproduced.** The mean accuracy over the five subjects is 44.4% (contiguous folds,
Euclidean), 53.4% (contiguous, density), 46.6% (random, Euclidean) and 56.3% (random, density),
against 48.4 +/- 7.6%; three of the four readings are within tolerance, and all four exceed every
one of 1000 mean null accuracies (maxima 9.7 to 12.5%). Chance under this procedure, the mean of
the null, is 7.8 to 9.1%, close to the paper's 8.3%. Per subject the accuracy is 40.8 to 60.6%.

### att-weights-cortex (blocked)

> The human-presence dimension of the attention-task state space has its weights in face and body
> areas (FFA, OFA, EBA), MT+, LO, the frontal operculum and the inferior frontal sulcus; the
> vehicle-presence dimension in RSC, PPA, OPA, IPS, FEF and around the inferior frontal sulcus.

The criterion compares mean weights within the subjects' functional ROIs, which the paper drew
from functional localizer and retinotopic mapping data (Methods, Preprocessing and Visualization,
paragraph 2). The data set holds no ROI, localizer or retinotopy files (listing in the data check),
the paper deposits them nowhere else, and the code repository has none. A group atlas would be a
substitute for the paper's per-subject ROIs, not the input the paper relies on, so it is not
used. The claim is blocked as data_missing; nothing is computed for it here.

## Video game task

States with no TRs are dropped (flash in S7; Supplementary Figure 4 shows no flash weights for
that subject), so S7's state space has 8 dimensions. Because the JSD is unchanged by affine maps,
the CSI does not depend on the order or signs of the dimensions; the dataset's column order is used.

### game-csi

> In the video game task the behavioural states are distinctly and significantly separated in the
> 9-dimensional state space, with CSI 0.72 in subject S1 and 0.62 in S7.

In [ ]:
rng = np.random.default_rng(4)
exp_g = {"S1": (0.72, 0.072), "S7": (0.62, 0.062)}
game_csi = {}
for s in ss.GAME_SUBJECTS:
    X, names = ss.game_matrix(game[s]["X"], game[s]["labels"], "dataset")
    Z = game[s]["Z"]
    pos = ss.positions(Z, X)
    for reading in "abc":
        mem, nm, dropped = ss.game_members(X, names, reading)
        vals = [ss.csi(pos, mem, np.random.default_rng(k)) for k in range(10)]
        null = []
        for _ in range(N_PERM):
            Xp = X[rng.permutation(len(X))]
            memp, _, _ = ss.game_members(Xp, names, reading)
            null.append(ss.csi(ss.positions(Z, Xp), memp, rng))
        null = np.array(null)
        game_csi[s, reading] = np.mean(vals)
        e, tol = exp_g[s]
        print(f"{s} reading {reading}: {len(mem)} clusters (sizes {[int(m.sum()) for m in mem]}; dropped {dropped}); "
              f"CSI {np.mean(vals):.3f} (seed SD {np.std(vals):.3f}); expected {e} +/- {tol}, within: {abs(np.mean(vals)-e) <= tol}; "
              f"null mean {null.mean():.3f}, max {null.max():.3f}; above every null value: {np.mean(vals) > null.max()}")
print(f"elapsed {time.time()-T0:.0f} s")

**Result: reproduced under one reading of the clusters.** With one cluster per state from TRs
that have exactly one state (reading b), the CSI is 0.702 in S1 and 0.581 in S7, within tolerance
of 0.72 and 0.62, and both exceed every null value (null maxima 0.270 and 0.258). With every TR in
the cluster of each of its states (reading a), S1 is within (0.694) and S7 just outside (0.549,
tolerance down to 0.558). With the three groups of Figure 3A (reading c), both are lower (0.513 and
0.388). All six values exceed their nulls.

### game-mutual-negative-projection

> The mutually exclusive states engage close and safe explore each project negatively onto the
> other's dimension of the video game state space.

Readings of the claims file: sign (a) each dimension oriented so its own state's TRs have a
positive mean, (b) the diagonal of R positive; order of the variables in the QR (i) Table 1, (ii)
the data set's columns. Null: the rows of the state matrix permuted and refitted, 20,000 times
(shared with fig3b-state-relations), one-sided for a negative mean.

In [ ]:
N_PERM_G = 20000
rng = np.random.default_rng(5)
proj = {}
for s in ss.GAME_SUBJECTS:
    Z = game[s]["Z"]
    for order in ["table1", "dataset"]:
        X, names = ss.game_matrix(game[s]["X"], game[s]["labels"], order)
        perms = [rng.permutation(len(X)) for _ in range(N_PERM_G)]
        for sign in ["own", "diag"]:
            M = ss.projection_matrix(ss.positions(Z, X, sign), X)
            null = np.array([ss.projection_matrix(ss.positions(Z, X[p], sign), X[p]) for p in perms])
            proj[s, order, sign] = (M, null, names)
rows = []
for (s, order, sign), (M, null, names) in proj.items():
    i, j = names.index("engage close"), names.index("safe explore")
    p1 = (1 + (null[:, i, j] <= M[i, j]).sum()) / (N_PERM_G + 1)
    p2 = (1 + (null[:, j, i] <= M[j, i]).sum()) / (N_PERM_G + 1)
    rows.append(dict(subject=s, order=order, sign=sign, ec_on_se=M[i, j], p_ec_on_se=p1, se_on_ec=M[j, i], p_se_on_ec=p2,
                     both_negative_p05=(M[i, j] < 0) and (M[j, i] < 0) and p1 < 0.05 and p2 < 0.05))
mn = pd.DataFrame(rows)
print(mn.round(5).to_string(index=False))
print("\nreadings with both means negative and p < 0.05 in both subjects:")
print(mn.groupby(["order", "sign"]).both_negative_p05.sum().to_string())
print(f"elapsed {time.time()-T0:.0f} s")

**Result: reproduced, resting on an assumption about the order of the variables.** With the
variables in the order of the data set's columns, engage-close TRs lie at -1.187 (S1) and -0.191
(S7) on the safe-explore dimension, and safe-explore TRs at -0.399 (S1) and -0.548 (S7) on the
engage-close dimension, all with p = 5e-5, the bound of 20,000 permutations, under both sign
readings. With the order of Table 1, S1 holds (-0.697 and -0.068, p = 5e-5) but in S7 safe explore
lies at -0.018 on the engage-close dimension (p = 0.937), so the claim would hold in one subject
only. The sign readings give the same values.

### fig3b-state-relations

> States of the video game project at significant non-zero positions on other states' dimensions,
> an asymmetric pattern found in both subjects that differs between them.

For each ordered pair of different states, the mean position of the row state's TRs on the column
state's dimension, two-sided permutation p from the same 20,000 permutations; Bonferroni over (a)
the off-diagonal pairs or (b) every cell, at 0.01.

In [ ]:
res3b = {}
for (s, order, sign), (M, null, names) in proj.items():
    d = len(names)
    p = (1 + (np.abs(null) >= np.abs(M)[None]).sum(0)) / (N_PERM_G + 1)
    off = ~np.eye(d, dtype=bool)
    for bonf, ntest in [("a", d * (d - 1)), ("b", d * d)]:
        sig = (p < 0.01 / ntest) & off
        asym = int((sig != sig.T).sum() // 2)
        res3b[s, order, sign, bonf] = (sig, names)
        print(f"{s} order {order:7s} sign {sign:4s} Bonferroni {bonf} ({ntest} tests, threshold {0.01/ntest:.2e}): "
              f"{int(sig.sum())} significant off-diagonal pairs; pairs significant in one direction only: {asym}")
print(f"smallest attainable p: {1/(N_PERM_G+1):.2e}")
for order in ["table1", "dataset"]:
    for sign in ["own", "diag"]:
        for bonf in "ab":
            s1, n1 = res3b["S1", order, sign, bonf]
            s7, n7 = res3b["S7", order, sign, bonf]
            set1 = {(n1[i], n1[j]) for i, j in zip(*np.nonzero(s1))}
            set7 = {(n7[i], n7[j]) for i, j in zip(*np.nonzero(s7))}
            print(f"order {order:7s} sign {sign:4s} Bonferroni {bonf}: S1 {len(set1)}, S7 {len(set7)}, shared {len(set1 & set7)}, "
                  f"sets differ: {set1 != set7}")
M, null, names = proj["S1", "table1", "own"]
print("\nS1, Table 1 order, own sign: mean position of row state's TRs on column dimension")
print(pd.DataFrame(M, index=names, columns=names).round(2).to_string())
M, null, names = proj["S7", "table1", "own"]
print("\nS7, Table 1 order, own sign")
print(pd.DataFrame(M, index=names, columns=names).round(2).to_string())
for s in ss.GAME_SUBJECTS:
    sig, names = res3b[s, "table1", "own", "a"]
    print(f"\n{s} significant pairs (Table 1 order, own sign, Bonferroni a), row state -> column dimension:")
    print(pd.DataFrame(np.where(sig, "x", "."), index=names, columns=names).to_string())

**Result: reproduced.** Under every sign, order and Bonferroni reading, both subjects have
significant off-diagonal projections (S1 45 or 48 pairs, S7 20 or 32), some pairs are significant
in one direction only (10 to 20), and the two subjects' sets of significant pairs differ (12 to 26
shared). The figure's own matrix was not available to compare pair by pair.

### game-classification

> Held-out TRs of the video game task are classified into the nine behavioural states from their
> position in the learned state space with balanced accuracy 32.8 +/- 3.8% in S1 and 30.4 +/- 4.2%
> in S7, above the chance of 11%.

Folds and distance as in att-classification. Labels: (a) TRs with exactly one state only; (b)
every labelled TR, a prediction correct for each of its states it names, recall per state over
the TRs that have it. The state space is fitted on all training TRs with the full state matrix.
Classes with no TRs are left out; for the density rule, a class needs more training TRs than
dimensions, so classes with fewer than 2 x (dimensions + 1) TRs under the label reading are left
out of that rule (listed below). Null: training state rows permuted, 1000 times.

In [ ]:
rng = np.random.default_rng(6)
gc_rows, gc_null = [], {}
for s in ss.GAME_SUBJECTS:
    t = time.time()
    X, names = ss.game_matrix(game[s]["X"], game[s]["labels"], "dataset")
    n, d = X.shape
    nst = X.sum(1)
    for scheme in ["contig", "random"]:
        fs = ss.fold_scores(game[s]["G"], ss.folds(n, scheme, rng))
        for lab in "ab":
            use = (nst == 1) if lab == "a" else (nst >= 1)
            single = np.where(nst == 1, X.argmax(1), -1)
            for rule in ["euclid", "density"]:
                counts = X[use].sum(0)
                minc = 1 if rule == "euclid" else 2 * (d + 1)
                classes = [c for c in range(d) if counts[c] >= minc]
                left = [names[c] for c in range(d) if counts[c] < minc]
                # held-out TRs scored under this label reading, and which of the classes each has
                scored = [(use[te], X[te][use[te]][:, classes] > 0) for tr, te, Ztr, Zte in fs]
                def run(perm_rng=None):
                    accs = []
                    for (tr, te, Ztr, Zte), (m, H) in zip(fs, scored):
                        Xtr = X[tr] if perm_rng is None else X[tr][perm_rng.permutation(len(tr))]
                        ptr, pte = ss.project_fold(Ztr, Zte, Xtr)
                        if lab == "a":
                            ntr = Xtr.sum(1)
                            lab_tr = np.where(ntr == 1, Xtr.argmax(1), -1)
                            cl = [c for c in classes if (lab_tr == c).sum() > (d if rule == "density" else 0)]
                            pred = ss.nearest_cluster(ptr, lab_tr, pte[m], cl, rule)
                        else:
                            # one cluster per state from every training TR that has it
                            if rule == "euclid":
                                means = np.array([ptr[Xtr[:, c] > 0].mean(0) for c in classes])
                                pred = np.asarray(classes)[((pte[m][:, None] - means[None]) ** 2).sum(-1).argmin(1)]
                            else:
                                ll = np.column_stack([ss._logpdf(pte[m], *ss._fit_normal(ptr[Xtr[:, c] > 0])) for c in classes])
                                pred = np.asarray(classes)[ll.argmax(1)]
                        accs.append(ss.balanced_accuracy(H, pred, classes))
                    return np.array(accs)
                accs = run()
                null = np.array([run(rng).mean() for _ in range(N_PERM_CLF)])
                gc_null[s, scheme, lab, rule] = null
                gc_rows.append(dict(subject=s, folds=scheme, labels=lab, rule=rule, n_classes=len(classes), left_out=left,
                                    bal_acc=100 * accs.mean(), fold_sd=100 * accs.std(), chance=100 / len(classes),
                                    null_mean=100 * null.mean(), null_max=100 * null.max(),
                                    p=(1 + (null >= accs.mean()).sum()) / (N_PERM_CLF + 1)))
    print(f"{s} done in {time.time()-t:.0f} s")
gcl = pd.DataFrame(gc_rows)
exp_c = {"S1": (32.8, 3.8), "S7": (30.4, 4.2)}
gcl["within"] = [abs(r.bal_acc - exp_c[r.subject][0]) <= exp_c[r.subject][1] for r in gcl.itertuples()]
gcl["above_null"] = gcl.bal_acc > gcl.null_max
print(gcl.round(4).to_string(index=False))
print("\nreadings where both subjects are within tolerance and above every null value:")
both = gcl.groupby(["folds", "labels", "rule"]).apply(lambda g: bool((g.within & g.above_null).all()), include_groups=False)
print(both.to_string())
print(f"elapsed {time.time()-T0:.0f} s, peak memory {mem_gb():.1f} GB")

**Result: partially reproduced.** Every reading is far above its null (p <= 0.001; chance under
this procedure 10.8 to 16.4%, against the paper's 11%). S7's balanced accuracy is within the
paper's 30.4 +/- 4.2% with contiguous folds (26.4 to 33.4%) and above it with random folds (36.8 to
46.5%). S1's is above the paper's 32.8 +/- 3.8% under every reading (42.8 to 54.6%), so no reading
puts both subjects within tolerance. The S1 game data hold 65 min, where the paper reports 90 min
for one subject, so the data may not be those the paper analysed.

### game-weights-two-groups (blocked)

> The video game state dimensions fall into two groups, dead, round start, safe explore, unsafe
> explore and flash in TPJ, precuneus and prefrontal cortex (default-mode areas), and kill, engage
> close, engage far and run away in motor, premotor and supplementary motor areas, IPS, FEF and
> SEF; the weights are largely consistent across the two subjects.

The criterion needs region definitions the data set does not hold (the paper's functional ROIs,
which are not public), and for S1 the game data (73,594 voxels) match no mapper in the data set
(S1's mapper has 51,608 voxels), so S1's weights cannot be placed on the cortex with the files
given. Blocked as data_missing; nothing is computed for it here.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| claim | outcome |
|---|---|
| fig1-csi-s2 | reproduced (assumption: third axis as the code gives it) |
| att-csi-group | reproduced (same assumption) |
| supp-csi-per-subject | reproduced (10 of 12 values; all significant under reading a) |
| att-passive-lower | not reproduced (2 or 3 of 5 subjects at the bound reached) |
| att-classification | reproduced |
| att-weights-cortex | blocked, data_missing (the paper's ROIs are not public) |
| game-csi | reproduced (one reading of the clusters) |
| game-mutual-negative-projection | reproduced (assumption: data set's column order) |
| fig3b-state-relations | reproduced |
| game-classification | partially reproduced (S1 above the paper's value) |
| game-weights-two-groups | blocked, data_missing (ROIs; no mapper for S1's game voxels) |

How the analysis differs from the paper's:

- The attention claims over subjects use five of the paper's six subjects: S4 is not released.
- The state space is the paper's code, computed exactly in the space of the 24 PCs; the code
  itself is run on the full voxels for the reported attention CSIs (R3).
- The attention variable has no weight in the deposited data, so the third axis of the attention
  state space is set by rounding; three readings are shown and the outcomes rest on the code's.
- Permutations: 1000 (20,000 for the game projections), so p-values reach 1e-3 (5e-5), not the
  paper's 1e-5. How labels are permuted, how folds are formed, the cluster distance, and how
  multi-state TRs enter are the claims file's readings, each run.
- S7's game state space has 8 dimensions (no flash TRs).

What was found about the data set:

- The attentive conditions of the deconvolved data are z-scored per voxel within each condition,
  so the attention variable (coded -1, 0, +1 in `states.hdf`, a coding the description does not
  give) carries no mean difference between them.
- S1's video game data hold 1950 TRs (65 min), where the paper and the description say one
  subject played 90 min; S7's hold 44 min, as stated.
- S1's video game data have 73,594 voxels, a voxel set no mapper in the data set matches.
- No functional ROI, localizer or retinotopy data are deposited.
- S7's video game BOLD is not z-scored per voxel over the session (voxel SDs 0.094 to 1.009),
  unlike S1's.
- The state columns of the video game files are not in the order of Table 1.